# 02 — Preprocessing & SOSA Mapping

**Goal:** Clean the 3W dataset and map it to the SOSA/SSN ontology for TKG ingestion.

## Steps:
1. Load and merge all instances
2. Handle NaN and frozen variables
3. Align timestamps (clock skew)
4. Map sensors to SOSA vocabulary
5. Add bitemporal annotation (valid_time + transaction_time)
6. Export preprocessed data for Neo4j loader

In [ ]:
import os
import json
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, lit, to_timestamp, unix_timestamp, min as spark_min, max as spark_max
from pyspark.sql.types import StringType, IntegerType, DoubleType, TimestampType
from pathlib import Path
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("3W_Preprocessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

DATA_ROOT   = Path('../../data/UseCase2/3w_dataset')
OUTPUT_DIR  = Path('../../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENSORS = ['P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-JUS-CKGL','QGL']

# SOSA mapping (Decision 1 — justified by Haller et al. [16])
SOSA_MAPPING = {
    'P-PDG':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'downhole'},
    'P-TPT':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'subsea_christmas_tree'},
    'T-TPT':      {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'subsea_christmas_tree'},
    'P-MON-CKP':  {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'platform_upstream_pck'},
    'T-JUS-CKP':  {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'platform_downstream_pck'},
    'P-JUS-CKGL': {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'gas_lift_downstream'},
    'T-JUS-CKGL': {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'gas_lift_downstream'},
    'QGL':        {'observable_property': 'VolumetricFlowRate', 'unit': 'sm3_per_s', 'position': 'gas_lift_injection'},
}

# Physical order for PRECEDES relation (Decision 1)
# Justified by Vargas et al. (2019): PDG(downhole) → TPT(subsea) → PCK(surface)
SENSOR_ORDER = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'T-JUS-CKGL', 'QGL']

EVENT_TYPES = {0:'Normal',1:'Abrupt_BSW',2:'Spurious_DHSV',3:'Severe_Slugging',
               4:'Flow_Instability',5:'Rapid_Productivity_Loss',
               6:'Quick_Restriction_PCK',7:'Scaling_PCK',8:'Hydrate'}

print('✅ Spark session initialized')
print(f'   Spark version: {spark.version}')
print(f'   Output dir: {OUTPUT_DIR}')

✅ Config loaded
   Output dir: ..\..\data\processed


## 1. Load All Instances

In [ ]:
def load_all_instances_spark(data_root: Path):
    """
    Load all Parquet files using Spark - distributed and memory-efficient.
    Returns a Spark DataFrame instead of pandas.
    """
    if not data_root.exists():
        print('⚠️  Dataset non trovato')
        return None

    # Collect all parquet file paths
    parquet_files = []
    for event_dir in sorted(data_root.iterdir()):
        if not event_dir.is_dir() or not event_dir.name.isdigit():
            continue
        event_code = int(event_dir.name)
        for parquet_file in sorted(event_dir.glob('*.parquet')):
            parquet_files.append((str(parquet_file), event_code, parquet_file.stem))

    if not parquet_files:
        print('⚠️  No parquet files found')
        return None

    print(f'📁 Found {len(parquet_files)} parquet files to process')

    # Read all files with Spark
    # First, read one file to get schema
    sample_df = spark.read.parquet(parquet_files[0][0])
    print(f'📊 Schema detected with {len(sample_df.columns)} columns')

    # Read all files and add metadata columns
    dfs_with_metadata = []
    for file_path, event_code, instance_id in parquet_files:
        try:
            df = spark.read.parquet(file_path)
            df = df.withColumn('well_id', lit(instance_id)) \
                   .withColumn('event_type', lit(event_code)) \
                   .withColumn('instance_id', lit(instance_id))
            dfs_with_metadata.append(df)
        except Exception as e:
            print(f'  ⚠️  Error loading {Path(file_path).name}: {e}')

    if not dfs_with_metadata:
        return None

    # Union all DataFrames
    combined_df = dfs_with_metadata[0]
    for df in dfs_with_metadata[1:]:
        combined_df = combined_df.union(df)

    total_rows = combined_df.count()
    print(f'✅ Loaded {len(dfs_with_metadata)} instances, {total_rows:,} total observations')
    return combined_df

# Load data with Spark
df_raw = load_all_instances_spark(DATA_ROOT)
if df_raw is not None:
    print(f'📊 DataFrame has {df_raw.count()} rows and {len(df_raw.columns)} columns')
    df_raw.show(3)

✅ Loaded 2228 instances, 76,587,318 total observations
   ABER-CKGL  ABER-CKP  ESTADO-DHSV  ESTADO-M1  ESTADO-M2  ESTADO-PXO  \
0        NaN       NaN          1.0        1.0        0.0         0.0   
1        NaN       NaN          1.0        1.0        0.0         0.0   
2        NaN       NaN          1.0        1.0        0.0         0.0   

   ESTADO-SDV-GL  ESTADO-SDV-P  ESTADO-W1  ESTADO-W2  ...  QGL  T-JUS-CKP  \
0            0.0           1.0        1.0        0.0  ...  0.0   84.64463   
1            0.0           1.0        1.0        0.0  ...  0.0   84.63828   
2            0.0           1.0        1.0        0.0  ...  0.0   84.63194   

   T-MON-CKP  T-PDG     T-TPT  class  state                    well_id  \
0        NaN    0.0  119.0781   <NA>   <NA>  WELL-00001_20170201010207   
1        NaN    0.0  119.0781   <NA>   <NA>  WELL-00001_20170201010207   
2        NaN    0.0  119.0781   <NA>   <NA>  WELL-00001_20170201010207   

   event_type                instance_id  
0  

## 2. Handle NaN and Frozen Variables

In [ ]:
def flag_data_quality_spark(df):
    """
    Flag NaN and frozen variables using Spark - distributed processing.
    Works on Spark DataFrame instead of pandas.
    """
    from pyspark.sql.functions import countDistinct, when, col, lit, isnan

    df = df.withColumn('quality_flag', lit('ok'))

    # Flag NaN values for each sensor
    for sensor in SENSORS:
        if sensor in df.columns:
            df = df.withColumn('quality_flag',
                when(col(sensor).isNull() | isnan(col(sensor)), 'missing')
                .otherwise(col('quality_flag')))

    # Flag frozen variables (same value throughout instance)
    # This is more complex in Spark - we need to check per instance_id
    for sensor in SENSORS:
        if sensor in df.columns:
            # Group by instance_id and count distinct non-null values
            frozen_check = df.filter(col(sensor).isNotNull()) \
                            .groupBy('instance_id') \
                            .agg(countDistinct(sensor).alias('distinct_count'),
                                 count(sensor).alias('total_count')) \
                            .filter((col('distinct_count') == 1) & (col('total_count') > 0)) \
                            .select('instance_id')

            # Get list of frozen instance_ids
            frozen_instances = [row['instance_id'] for row in frozen_check.collect()]

            if frozen_instances:
                # Update quality_flag for frozen instances
                df = df.withColumn('quality_flag',
                    when(col('instance_id').isin(frozen_instances), 'frozen')
                    .otherwise(col('quality_flag')))

    # Count quality flags
    quality_counts = df.groupBy('quality_flag').count().orderBy('quality_flag')
    print('Data quality flags:')
    quality_counts.show()

    return df

if df_raw is not None:
    df_flagged = flag_data_quality_spark(df_raw)
    print(f'✅ Quality flagging completed')
else:
    print('⚠️  Load dataset first')

ArrowMemoryError: malloc of size 1465666688 failed

## 3. Add Bitemporal Annotation

In [ ]:
def add_bitemporality_spark(df):
    """
    Add bitemporal annotation using Spark.
    valid_time = when the sensor reading is true in the real world
    tx_time = when it was ingested into the TKG
    """
    from pyspark.sql.functions import date_format, current_timestamp

    # Convert timestamp to proper format and add bitemporal columns
    df = df.withColumn('valid_from',
        date_format(col('timestamp'), 'yyyy-MM-dd HH:mm:ss')) \
           .withColumn('valid_to', lit(None).cast(StringType())) \
           .withColumn('tx_time',
               date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss'))

    # Get timestamp range
    time_range = df.agg(
        spark_min('valid_from').alias('min_time'),
        spark_max('valid_from').alias('max_time')
    ).collect()[0]

    print('✅ Bitemporal annotation added')
    print(f'   valid_from range: {time_range["min_time"]} → {time_range["max_time"]}')

    return df

if df_raw is not None:
    df_bitemporal = add_bitemporality_spark(df_flagged)
    print(f'✅ Bitemporal annotation completed')
else:
    print('⚠️  Load dataset first')

## 4. Export Preprocessed Data

In [ ]:
def export_for_neo4j_spark(df, output_dir: Path):
    """
    Export preprocessed data in format ready for Neo4j loader using Spark.
    Saves: observations.parquet + sosa_mapping.json
    """
    # Save observations as Parquet (Spark native format)
    output_path = str(output_dir / 'observations_3w_spark')
    df.write.mode('overwrite').parquet(output_path)
    print(f'✅ Observations saved: {output_path}')

    # Get row count (efficient in Spark)
    row_count = df.count()
    col_count = len(df.columns)
    print(f'   Shape: ({row_count:,} rows, {col_count} columns)')

    # Save SOSA mapping (same as before)
    mapping_path = output_dir / 'sosa_mapping.json'
    with open(mapping_path, 'w') as f:
        json.dump(SOSA_MAPPING, f, indent=2)
    print(f'✅ SOSA mapping saved: {mapping_path}')

    # Save sensor order for PRECEDES relation
    order_path = output_dir / 'sensor_order.json'
    with open(order_path, 'w') as f:
        json.dump({'precedes_order': SENSOR_ORDER,
                   'justification': 'Physical order: downhole → subsea → surface (Vargas et al. 2019)'}, f, indent=2)
    print(f'✅ Sensor order saved: {order_path}')

if df_raw is not None:
    export_for_neo4j_spark(df_bitemporal, OUTPUT_DIR)
    print('✅ Export completed - ready for Neo4j ingestion')
else:
    print('⚠️  Load dataset first')

# Clean up Spark session
spark.stop()
print('🧹 Spark session stopped')